# Observations

> Several functions to extract different kinds of observations from the environment. These are used by the different observation builders to construct the final observation that is returned to the agent.

In [ ]:
#| default_exp envs.multigrid.obs

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Any


In [ ]:
#| export
import numba as nb
import numpy as np

from stable_marl.envs.multigrid.core.agent import AgentState, Agent
from stable_marl.envs.multigrid.core.constants import Color, Direction, State, Type
from stable_marl.envs.multigrid.core.world_object import Wall, WorldObj
from stable_marl.envs.multigrid.core.grid import Grid

from numpy.typing import NDArray as ndarray



In [ ]:
#| export

WALL_ENCODING = Wall().encode()
UNSEEN_ENCODING = WorldObj(Type.unseen, Color.from_index(0)).encode()
ENCODE_DIM = WorldObj.dim

GRID_ENCODING_IDX = slice(None)

AGENT_DIR_IDX = AgentState.DIR
AGENT_POS_IDX = AgentState.POS
AGENT_TERMINATED_IDX = AgentState.TERMINATED
AGENT_CARRYING_IDX = AgentState.CARRYING
AGENT_ENCODING_IDX = AgentState.ENCODING

TYPE = WorldObj.TYPE
STATE = WorldObj.STATE

WALL = int(Type.wall)
DOOR = int(Type.door)

OPEN = int(State.open)
CLOSED = int(State.closed)
LOCKED = int(State.locked)

RIGHT = int(Direction.right)
LEFT = int(Direction.left)
UP = int(Direction.up)
DOWN = int(Direction.down)




In [ ]:
#| export

@nb.njit(cache=True)
def see_behind(
    world_obj: ndarray[np.int_] # World object encoding
) -> bool:
    "Can an agent see behind this object?"
    if world_obj is None:
        return True
    if world_obj[TYPE] == WALL:
        return False
    elif world_obj[TYPE] == DOOR and world_obj[STATE] != OPEN:
        return False

    return True

In [ ]:
#| export

@nb.njit(cache=True)
def get_view_exts(
    agent_dir: ndarray[np.int_], # Direction of each agent
    agent_pos: ndarray[np.int_], # The (x, y) position of each agent
    agent_view_size: int # Width and height of agent view
) -> ndarray[np.int_]: # The (x, y) coordinates of the top-left corner of each agent's observable view
    "Get the extents of the square set of grid cells visible to each agent."
    agent_x, agent_y = agent_pos[:, 0], agent_pos[:, 1]
    top_left = np.zeros((agent_dir.shape[0], 2), dtype=np.int_)

    # Facing right
    top_left[agent_dir == RIGHT, 0] = agent_x[agent_dir == RIGHT]
    top_left[agent_dir == RIGHT, 1] = agent_y[agent_dir == RIGHT] - agent_view_size // 2

    # Facing down
    top_left[agent_dir == DOWN, 0] = agent_x[agent_dir == DOWN] - agent_view_size // 2
    top_left[agent_dir == DOWN, 1] = agent_y[agent_dir == DOWN]

    # Facing left
    top_left[agent_dir == LEFT, 0] = agent_x[agent_dir == LEFT] - agent_view_size + 1
    top_left[agent_dir == LEFT, 1] = agent_y[agent_dir == LEFT] - agent_view_size // 2

    # Facing up
    top_left[agent_dir == UP, 0] = agent_x[agent_dir == UP] - agent_view_size // 2
    top_left[agent_dir == UP, 1] = agent_y[agent_dir == UP] - agent_view_size + 1

    return top_left


In [ ]:
#| export
@nb.njit(cache=True)
def get_see_behind_mask(
    grid_array: ndarray[np.int_] # Grid object array for each agent
) -> ndarray[np.int_]: # Boolean visibility mask
    "Return boolean mask indicating which grid locations can be seen through."
    num_agents, width, height = grid_array.shape[:3]
    see_behind_mask = np.zeros((num_agents, width, height), dtype=np.bool_)
    for agent in range(num_agents):
        for i in range(width):
            for j in range(height):
                see_behind_mask[agent, i, j] = see_behind(grid_array[agent, i, j])

    return see_behind_mask


In [ ]:
#| export

@nb.njit(cache=True)
def get_vis_mask(
    obs_grid: ndarray[np.int_] # Grid object array for each agent observation
) -> ndarray[np.bool_]: # Boolean visibility mask for each agent
    "Generate a boolean mask indicating which grid locations are visible to each agent."
    num_agents, width, height = obs_grid.shape[:3]
    see_behind_mask = get_see_behind_mask(obs_grid)
    vis_mask = np.zeros((num_agents, width, height), dtype=np.bool_)
    vis_mask[:, width // 2, height - 1] = True # agent relative position

    for agent in range(num_agents):
        for j in range(height - 1, -1, -1):
            # Forward pass
            for i in range(0, width - 1):
                if vis_mask[agent, i, j] and see_behind_mask[agent, i, j]:
                    vis_mask[agent, i + 1, j] = True
                    if j > 0:
                        vis_mask[agent, i + 1, j - 1] = True
                        vis_mask[agent, i, j - 1] = True

            # Backward pass
            for i in range(width - 1, 0, -1):
                if vis_mask[agent, i, j] and see_behind_mask[agent, i, j]:
                    vis_mask[agent, i - 1, j] = True
                    if j > 0:
                        vis_mask[agent, i - 1, j - 1] = True
                        vis_mask[agent, i, j - 1] = True

    return vis_mask


In [ ]:
#| export
@nb.njit(cache=True)
def gen_obs_grid(
    grid_state: ndarray[np.int_], # Array representation for each grid object
    agent_state: ndarray[np.int_], # Array representation for each agent
    agent_view_size: int # Width and height of observation sub-grids
) -> ndarray[np.int_]: # Observed sub-grid for each agent
    "Generate the sub-grid observed by each agent (WITHOUT visibility mask)."
    num_agents = len(agent_state)
    obs_width, obs_height = agent_view_size, agent_view_size

    # Process agent states
    agent_grid_encoding = agent_state[..., AGENT_ENCODING_IDX]
    agent_dir = agent_state[..., AGENT_DIR_IDX]
    agent_pos = agent_state[..., AGENT_POS_IDX]
    agent_terminated = agent_state[..., AGENT_TERMINATED_IDX]
    agent_carrying_encoding = agent_state[..., AGENT_CARRYING_IDX]

    # Get grid encoding
    if num_agents > 1:
        grid_encoding = np.empty((*grid_state.shape[:-1], ENCODE_DIM), dtype=np.int_)
        grid_encoding[...] = grid_state[..., GRID_ENCODING_IDX]

        # Insert agent grid encodings
        for agent in range(num_agents):
            if not agent_terminated[agent]:
                i, j = agent_pos[agent]
                grid_encoding[i, j, GRID_ENCODING_IDX] = agent_grid_encoding[agent]
    else:
        grid_encoding = grid_state[..., GRID_ENCODING_IDX]

    # Get top left corner of observation grids
    top_left = get_view_exts(agent_dir, agent_pos, agent_view_size)
    topX, topY = top_left[:, 0], top_left[:, 1]

    # Populate observation grids
    num_left_rotations = (agent_dir + 1) % 4
    obs_grid = np.empty((num_agents, obs_width, obs_height, ENCODE_DIM), dtype=np.int_)
    for agent in range(num_agents):
        for i in range(0, obs_width):
            for j in range(0, obs_height):
                # Absolute coordinates in world grid
                x, y = topX[agent] + i, topY[agent] + j

                # Rotated relative coordinates for observation grid
                if num_left_rotations[agent] == 0:
                    i_rot, j_rot = i, j
                elif num_left_rotations[agent] == 1:
                    i_rot, j_rot = j, obs_width - i - 1
                elif num_left_rotations[agent] == 2:
                    i_rot, j_rot = obs_width - i - 1, obs_height - j - 1
                elif num_left_rotations[agent] == 3:
                    i_rot, j_rot = obs_height - j - 1, i

                # Set observation grid
                if 0 <= x < grid_encoding.shape[0] and 0 <= y < grid_encoding.shape[1]:
                    obs_grid[agent, i_rot, j_rot] = grid_encoding[x, y]
                else:
                    obs_grid[agent, i_rot, j_rot] = WALL_ENCODING

    # Make it so each agent sees what it's carrying
    # We do this by placing the carried object at the agent position
    # in each agent's partially observable view
    obs_grid[:, obs_width // 2, obs_height - 1] = agent_carrying_encoding

    return obs_grid


In [ ]:
#| export

@nb.njit(cache=True)
def gen_obs_grid_encoding(
    grid_state: ndarray[np.int_], # Array representation for each grid object
    agent_state: ndarray[np.int_], # Array representation for each agent
    agent_view_size: int, # Width and height of observation sub-grids
    see_through_walls: bool # Whether the agent can see through walls
) -> ndarray[np.int_]: # Encoding of observed sub-grid for each agent
    "Generate encoding for the sub-grid observed by an agent (including visibility mask)."
    obs_grid = gen_obs_grid(grid_state, agent_state, agent_view_size)

    # Generate and apply visibility masks
    vis_mask = get_vis_mask(obs_grid)
    num_agents = len(agent_state)
    for agent in range(num_agents):
        if not see_through_walls:
            for i in range(agent_view_size):
                for j in range(agent_view_size):
                    if not vis_mask[agent, i, j]:
                        obs_grid[agent, i, j] = UNSEEN_ENCODING

    return obs_grid



In [ ]:
#| export

@nb.njit(cache=True)
def get_see_behind_mask(
    grid_array: ndarray[np.int_] # Grid object array for each agent
) -> ndarray[np.int_]: # Boolean visibility mask
    "Return boolean mask indicating which grid locations can be seen through."
    num_agents, width, height = grid_array.shape[:3]
    see_behind_mask = np.zeros((num_agents, width, height), dtype=np.bool_)
    for agent in range(num_agents):
        for i in range(width):
            for j in range(height):
                see_behind_mask[agent, i, j] = see_behind(grid_array[agent, i, j])

    return see_behind_mask


In [ ]:
#| export

def get_agent_obs_grid(
    grid: Grid,
    agent: Agent,
    agent_view_size: int,
) -> Grid:
    agent_dir = np.array([agent.state.dir.value])
    agent_pos = np.array([agent.state.pos])

    top_left = get_view_exts(agent_dir, agent_pos, agent_view_size)
    topX, topY = top_left[0, 0], top_left[0, 1]

    num_left_rotations = (agent_dir[0] + 1) % 4
    obs_grid = Grid(agent_view_size, agent_view_size)

    for i in range(agent_view_size):
        for j in range(agent_view_size):
            x, y = topX + i, topY + j

            if num_left_rotations == 0:
                i_rot, j_rot = i, j
            elif num_left_rotations == 1:
                i_rot, j_rot = j, agent_view_size - i - 1
            elif num_left_rotations == 2:
                i_rot, j_rot = agent_view_size - i - 1, agent_view_size - j - 1
            elif num_left_rotations == 3:
                i_rot, j_rot = agent_view_size - j - 1, i

            if 0 <= x < grid.width and 0 <= y < grid.height:
                obs_grid.set(i_rot, j_rot, grid.get(x, y))
            else:
                obs_grid.set(i_rot, j_rot, Wall())


    return obs_grid


In [ ]:
#| export
def gen_obs_grid_image(
    grid: Grid,
    agents: list[Agent],
    agents_states: AgentState,
    agent_view_size: int,
    tile_size: int = 32,
    see_through_walls: bool | None = None,
) -> ndarray:
    "Render the RGB point-of-view image of each agent."
    num_agents = len(agents)

    # Visibility masks, computed from each agent's own state
    # (agents may be temporary copies, e.g. in get_goal_state)
    own_states = np.stack([agent.state._view for agent in agents])
    vis_masks = get_vis_mask(gen_obs_grid(grid.state, own_states, agent_view_size))
    obs_images = np.zeros(
        (num_agents, agent_view_size * tile_size, agent_view_size * tile_size, 3),
        dtype=np.uint8
    )

    # obs_grid_encoded = gen_obs_grid(grid.state, agents_states, agent_view_size)
    # vis_mask = get_vis_mask(obs_grid_encoded)
    # unseen_obj = WorldObj(Type.unseen, Color.from_index(0))
    for agent_idx, agent in enumerate(agents):
        
        obs_grid = get_agent_obs_grid(grid,
                                      agent,
                                      agent_view_size)
        
        # if not see_through_walls:
        #     # Apply the vis_mask from the encoded pipeline directly
        #     for i in range(agent_view_size):
        #         for j in range(agent_view_size):
        #             if not vis_mask[agent_idx, i, j]:
        #                 obs_grid.set(i, j, unseen_obj)
            
        # Other (non-terminated) agents inside the view, in view coordinates
        # (same rule as `gen_obs_grid`, which only encodes non-terminated agents)
        view_agents = []
        top_x, top_y = get_view_exts(
            np.array([agent.state.dir]), np.array([agent.state.pos]), agent_view_size)[0]
        num_left_rotations = (agent.state.dir + 1) % 4
        for other in agents:
            if other is agent or other.state.terminated:
                continue
            i, j = other.state.pos[0] - top_x, other.state.pos[1] - top_y
            if not (0 <= i < agent_view_size and 0 <= j < agent_view_size):
                continue
            for _ in range(num_left_rotations): # rotate like `get_agent_obs_grid`
                i, j = j, agent_view_size - i - 1
            view_agent = Agent(index=other.index)
            view_agent.state.pos = (i, j)
            view_agent.state.dir = (other.state.dir - num_left_rotations) % 4
            view_agent.state.color = other.state.color.name
            view_agents.append(view_agent)

        # Place ego agent at bottom center facing up (last, so it is drawn on top)
        ego_agent = Agent(index=agent_idx)
        ego_agent.state.pos = np.array([agent_view_size // 2, agent_view_size - 1])
        ego_agent.state.dir = Direction.up
        ego_agent.state.color = agent.state.color.name

        obs_images[agent_idx] = obs_grid.render(tile_size, agents=[*view_agents, ego_agent])

        # Black out cells the agent cannot see
        agent_see_through = agent.see_through_walls if see_through_walls is None else see_through_walls
        if not agent_see_through:
            for i, j in np.argwhere(~vis_masks[agent_idx]):
                obs_images[agent_idx,
                           j * tile_size:(j + 1) * tile_size,
                           i * tile_size:(i + 1) * tile_size] = 0

    return obs_images

Cells hidden from an agent (same visibility rule as `gen_obs_grid_encoding`)
are blacked out, unless the agent can see through walls.
`see_through_walls=None` uses each agent's own `see_through_walls` setting.

In [ ]:
#| export
def _world_encoding(
    grid_state: ndarray[np.int_],
    agent_state: ndarray[np.int_]) -> ndarray[np.int_]:
    """
    Grid encoding of shape (width, height, encode_dim) with all
    non-terminated agents inserted (same rule as `gen_obs_grid`).
    """
    agent_state = np.asarray(agent_state)
    encoding = np.array(grid_state[..., GRID_ENCODING_IDX])
    for state in agent_state:
        if not state[AGENT_TERMINATED_IDX]:
            x, y = state[AGENT_POS_IDX]
            encoding[x, y] = state[AGENT_ENCODING_IDX]
    return encoding


def gen_obs_grid_allo(
    grid_state: ndarray[np.int_],
    agent_state: ndarray[np.int_],
    agent_view_size: int) -> ndarray[np.int_]:
    """
    Generate the allocentric sub-grid observed by each agent (WITHOUT visibility mask):
    a world-aligned (north up, no rotation) window centred on the agent.

    Returns: Indexed [i, j] like the world grid; the agent is at (view_size // 2, view_size // 2)
    """
    agent_state = np.asarray(agent_state)
    encoding = _world_encoding(grid_state, agent_state)
    width, height = encoding.shape[:2]
    r = agent_view_size // 2

    obs_grid = np.empty((len(agent_state), agent_view_size, agent_view_size, ENCODE_DIM), dtype=np.int_)
    for agent, state in enumerate(agent_state):
        x, y = state[AGENT_POS_IDX]
        xs, ys = np.arange(x - r, x + r + 1), np.arange(y - r, y + r + 1)
        valid_x, valid_y = (0 <= xs) & (xs < width), (0 <= ys) & (ys < height)
        obs_grid[agent] = WALL_ENCODING # out of bounds cells are walls (as in `gen_obs_grid`)
        obs_grid[agent][np.ix_(valid_x, valid_y)] = encoding[np.ix_(xs[valid_x], ys[valid_y])]

    return obs_grid


def get_vis_mask_allo(obs_grid: ndarray[np.int_]) -> ndarray[np.bool_]:
    """
    Visibility mask for allocentric (agent-centred) observation grids.

    Returns: ndarray[bool] of shape (num_agents, view_size, view_size)
    """
    num_agents, view_size = obs_grid.shape[:2]
    half = view_size // 2 + 1 # rows from the window edge up to the agent's row
    vis_mask = np.zeros((num_agents, view_size, view_size), dtype=bool)
    for k in range(4):
        rotated = np.rot90(obs_grid, k, axes=(1, 2))
        half_mask = np.zeros((num_agents, view_size, view_size), dtype=bool)
        half_mask[:, :, :half] = get_vis_mask(np.ascontiguousarray(rotated[:, :, :half]))
        vis_mask |= np.rot90(half_mask, -k, axes=(1, 2))

    return vis_mask


def gen_obs_grid_encoding_allo(
    grid_state: ndarray[np.int_],
    agent_state: ndarray[np.int_],
    agent_view_size: int,
    see_through_walls: bool) -> ndarray[np.int_]:
    """
    Allocentric encoding of the sub-grid observed by each agent (including visibility mask).

    Returns: ndarray[int] of shape (num_agents, view_size, view_size, encode_dim)
    """
    obs_grid = gen_obs_grid_allo(grid_state, agent_state, agent_view_size)
    if not see_through_walls:
        obs_grid[~get_vis_mask_allo(obs_grid)] = UNSEEN_ENCODING

    return obs_grid


def gen_obs_grid_encoding_global(
    grid_state: ndarray[np.int_],
    agent_state: ndarray[np.int_]) -> ndarray[np.int_]:
    """
    Global (fully observable) encoding of the whole grid, identical for every agent.

    Returns: ndarray[int] of shape (num_agents, width, height, encode_dim)
    """
    encoding = _world_encoding(grid_state, agent_state)
    return np.repeat(encoding[None], len(agent_state), axis=0)


def _copy_agent(agent: Agent, pos, dir) -> Agent:
    """
    Lightweight copy of an agent (for rendering only) at a given position / direction.
    """
    copy = Agent(index=agent.index)
    copy.state.pos = pos
    copy.state.dir = dir
    copy.state.color = agent.state.color.name
    return copy


def gen_obs_grid_image_allo(
    grid: Grid,
    agents: list[Agent],
    agent_view_size: int,
    tile_size: int = 32,
    see_through_walls: bool | None = None,
) -> ndarray:
    """
    Render the allocentric RGB view of each agent: a world-aligned window
    centred on the agent, with every agent drawn at its true heading.
    """
    r = agent_view_size // 2
    own_states = np.stack([agent.state._view for agent in agents])
    vis_masks = get_vis_mask_allo(gen_obs_grid_allo(grid.state, own_states, agent_view_size))
    obs_images = np.zeros(
        (len(agents), agent_view_size * tile_size, agent_view_size * tile_size, 3),
        dtype=np.uint8
    )

    for agent_idx, agent in enumerate(agents):
        x0, y0 = agent.state.pos[0] - r, agent.state.pos[1] - r

        # World-aligned window (out of bounds cells are walls)
        obs_grid = Grid(agent_view_size, agent_view_size)
        for i in range(agent_view_size):
            for j in range(agent_view_size):
                x, y = x0 + i, y0 + j
                if 0 <= x < grid.width and 0 <= y < grid.height:
                    obs_grid.set(i, j, grid.get(x, y))
                else:
                    obs_grid.set(i, j, Wall())

        # Other non-terminated agents in the window, then the agent itself (drawn on top)
        view_agents = []
        for other in agents:
            if other is agent or other.state.terminated:
                continue
            i, j = other.state.pos[0] - x0, other.state.pos[1] - y0
            if 0 <= i < agent_view_size and 0 <= j < agent_view_size:
                view_agents.append(_copy_agent(other, (i, j), other.state.dir))
        view_agents.append(_copy_agent(agent, (r, r), agent.state.dir))

        obs_images[agent_idx] = obs_grid.render(tile_size, agents=view_agents)

        # Black out cells the agent cannot see
        agent_see_through = agent.see_through_walls if see_through_walls is None else see_through_walls
        if not agent_see_through:
            for i, j in np.argwhere(~vis_masks[agent_idx]):
                obs_images[agent_idx,
                           j * tile_size:(j + 1) * tile_size,
                           i * tile_size:(i + 1) * tile_size] = 0

    return obs_images


def gen_obs_grid_image_global(
    grid: Grid,
    agents: list[Agent],
    tile_size: int = 32,
) -> ndarray:
    """
    Render the whole grid (fully observable) with all non-terminated agents,
    identical for every agent.

    Returns: ndarray[uint8] of shape (num_agents, height * tile_size, width * tile_size, 3)
    """
    image = grid.render(tile_size, agents=[a for a in agents if not a.state.terminated])
    return np.repeat(image[None], len(agents), axis=0)


#### `get_vis_mask_allo`

Applies the egocentric `get_vis_mask` rule to the four half-windows
(one per direction) around the centre, and combines them.

#### `gen_obs_grid_image_allo`

Hidden cells are blacked out unless the agent can see through walls.
`see_through_walls=None` uses each agent's own `see_through_walls` setting.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()